In [1]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# ---------- 1. Page-level classifier (gives emission probabilities) ----------
data = [
    ("invoice number total amount due tax gst payment terms", "invoice"),
    ("bill to customer quantity unit price subtotal payable", "invoice"),
    ("tax invoice item description amount bank account details", "invoice"),
    ("invoice balance due purchase order number remit payment", "invoice"),
    ("curriculum vitae objective education skills python experience", "resume"),
    ("work experience software engineer projects certifications", "resume"),
    ("resume career summary academic qualifications technical skills", "resume"),
    ("professional experience education degree languages hobbies", "resume"),
    ("patient name diagnosis blood pressure prescription doctor hospital", "medical"),
    ("lab report hemoglobin glucose cholesterol test results patient", "medical"),
    ("discharge summary patient admitted medication dosage follow up", "medical"),
    ("clinical report physician prescribed tablets patient", "medical"),
    ("this agreement is made between the parties hereinafter terms", "legal"),
    ("contract clause liability indemnity jurisdiction governing law witness", "legal"),
    ("non disclosure agreement confidential information party obligations", "legal"),
    ("lease agreement tenant landlord rent security deposit termination", "legal"),
]
texts, labels = zip(*data)

clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("lr", LogisticRegression(max_iter=1000, C=10)),
])
clf.fit(texts, labels)
classes = list(clf.named_steps["lr"].classes_)
K = len(classes)

# ---------- 2. Probabilistic layer: HMM-style Viterbi decoding ----------
# Transition matrix: documents usually continue (high self-transition)
stay = 0.80
A = np.full((K, K), (1 - stay) / (K - 1))
np.fill_diagonal(A, stay)
logA = np.log(A)
log_pi = np.log(np.full(K, 1.0 / K))

def viterbi(emissions):
    """emissions: (T, K) classifier probabilities -> most likely state path."""
    T = emissions.shape[0]
    logE = np.log(emissions + 1e-9)
    dp = np.zeros((T, K))
    bp = np.zeros((T, K), dtype=int)
    dp[0] = log_pi + logE[0]
    for t in range(1, T):
        scores = dp[t - 1][:, None] + logA          # (prev, curr)
        bp[t] = scores.argmax(axis=0)
        dp[t] = scores.max(axis=0) + logE[t]
    path = [int(dp[-1].argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(bp[t][path[-1]]))
    return path[::-1]

# ---------- 3. FSM layer: boundary detection ----------
# Rule-based first-page cues
START_CUES = re.compile(
    r"(invoice number|tax invoice|curriculum vitae|resume|patient name|"
    r"discharge summary|lab report|this agreement is made|affidavit)", re.I)

# States: IDLE -> IN_DOC <-> NEW_DOC
# Events: LABEL_CHANGE, CUE, CONTINUE
FSM = {
    ("IDLE",   "LABEL_CHANGE"): "NEW_DOC",
    ("IDLE",   "CUE"):          "NEW_DOC",
    ("IDLE",   "CONTINUE"):     "NEW_DOC",   # first page always opens a document
    ("IN_DOC", "LABEL_CHANGE"): "NEW_DOC",
    ("IN_DOC", "CUE"):          "NEW_DOC",
    ("IN_DOC", "CONTINUE"):     "IN_DOC",
    ("NEW_DOC","LABEL_CHANGE"): "NEW_DOC",
    ("NEW_DOC","CUE"):          "IN_DOC",    # cue on page right after a boundary: ignore
    ("NEW_DOC","CONTINUE"):     "IN_DOC",
}

def fsm_segment(pages, state_path, min_pages=1):
    state, boundaries, prev_label = "IDLE", [], None
    for i, (page, k) in enumerate(zip(pages, state_path)):
        label = classes[k]
        if prev_label is None:
            event = "CONTINUE"
        elif label != prev_label:
            event = "LABEL_CHANGE"
        elif START_CUES.search(page):
            event = "CUE"
        else:
            event = "CONTINUE"

        state = FSM[(state, event)]
        if state == "NEW_DOC":
            # enforce minimum document length (suppress spurious boundaries)
            if not boundaries or (i - boundaries[-1]) >= min_pages:
                boundaries.append(i)
            else:
                state = "IN_DOC"
        prev_label = label
    return boundaries

# ---------- 4. Test batch (one noisy page inside the resume) ----------
pages = [
    "Invoice number 9001 total amount due GST payment terms",
    "Item description quantity unit price subtotal tax amount payable",
    "Curriculum vitae objective education skills python experience",
    "Work experience projects certifications skills",
    "Total amount payable tax",                       # noisy page inside the resume
    "Professional experience internship education degree references",
    "Patient name diagnosis prescription doctor hospital treatment",
    "Lab report glucose cholesterol test results patient",
    "This agreement is made between the parties terms clause liability",
    "Governing law jurisdiction indemnity witness signed parties",
]

emissions = clf.predict_proba(pages)
raw_labels = [classes[i] for i in emissions.argmax(axis=1)]
vit_path = viterbi(emissions)
vit_labels = [classes[i] for i in vit_path]
bounds = fsm_segment(pages, vit_path, min_pages=2)

print(f"{'Page':<6}{'Raw classifier':<16}{'After Viterbi':<16}")
for i in range(len(pages)):
    print(f"{i+1:<6}{raw_labels[i]:<16}{vit_labels[i]:<16}")

print("\nDetected document boundaries (page numbers where a new document starts):")
print([b + 1 for b in bounds])

print("\nSegmented documents:")
bounds_end = bounds + [len(pages)]
for n in range(len(bounds)):
    s, e = bounds_end[n], bounds_end[n + 1]
    types = [vit_labels[i] for i in range(s, e)]
    dtype = max(set(types), key=types.count)
    print(f"  Document {n+1}: type = {dtype:<8} pages = {list(range(s+1, e+1))}")

Page  Raw classifier  After Viterbi   
1     invoice         invoice         
2     invoice         invoice         
3     resume          resume          
4     resume          resume          
5     invoice         resume          
6     resume          resume          
7     medical         medical         
8     medical         medical         
9     legal           legal           
10    legal           legal           

Detected document boundaries (page numbers where a new document starts):
[1, 3, 7, 9]

Segmented documents:
  Document 1: type = invoice  pages = [1, 2]
  Document 2: type = resume   pages = [3, 4, 5, 6]
  Document 3: type = medical  pages = [7, 8]
  Document 4: type = legal    pages = [9, 10]
